# Day 2: Why These 3 Frameworks

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> Not all frameworks are the same. They answer the same question differently.

Yesterday you built the same agent three ways. Today we stop and ask: **WHY these three?**
Why not LangChain? Why not LlamaIndex? Why not Microsoft AutoGen?

This is the day where a senior developer would sit you down and explain the
landscape before you dive deeper. Understanding the **mental model** behind each
framework will save you weeks of confusion later.

## What You'll Learn
- The mental model behind each framework
- Why we chose these 3 (and dropped the others)
- How to read framework documentation like a senior developer
- A code tour of each framework's internals

In [4]:
# ── Setup ───────────────────────────────────────────────

import sys
sys.path.insert(0, "..")

from shared import check_and_report, print_config, disable_openai_tracing
from shared import get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()

disable_openai_tracing()
print("=" * 50)
print("DAY 2 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (8 model(s) available)

DAY 2 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


---
## The Big Picture: Three Mental Models

Every framework makes a bet on how developers should think about agents.
Here are the three bets:

| Framework | Mental Model | In One Sentence |
|---|---|---|
| **OpenAI Agents SDK** | Agent = **Worker** | Give a worker a job, tools, and rules. | |
| **LangGraph** | Agent = **Flowchart** | Draw nodes and edges. The LLM decides which path. | |
| **CrewAI** | Agent = **Team Member** | Define roles, goals, and backstories. They collaborate. |

These aren't just marketing slogans. They affect **every line of code** you write.

Let's prove it by building the same "calculate 15% of 2340" agent in all three.

## Framework 1: OpenAI Agents SDK — The Worker Model

**Why we chose it:** It's the fastest path to a working agent. OpenAI built it for
their own API, so the integration is seamless. When someone asks "how do I build an
agent?" — this is the simplest answer.

**The mental model:** You're a manager hiring a worker. You give them:
- A **name** (identity)
- **Instructions** (job description)
- **Tools** (what they can use)
- **Handoffs** (when to transfer to another worker)

**When it wins:** Quick prototypes, single-agent tools, simple handoffs.
**When it loses:** Complex workflows with loops, branching, and state persistence.

In [5]:
# ── OpenAI Agents SDK: The Worker Model ─────────────────
# Notice how this reads like a job description.

from agents import Agent, Runner, function_tool

model = get_openai_agents_model()

# A tool — just a Python function with @function_tool
@function_tool
def calculate(expression: str) -> str:
    """Evaluate a math expression like '15% of 2340'."""
    import re
    pct_match = re.match(r"(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)", expression)
    if pct_match:
        return str(float(pct_match.group(2)) * float(pct_match.group(1)) / 100)
    try:
        return str(eval(expression))
    except Exception:
        return f"Could not evaluate: {expression}"

# The worker: name + instructions + tools
calculator_agent = Agent(
    name="Calculator",
    instructions="You calculate math expressions. Use the calculate tool for all math.",
    tools=[calculate],
    model=model,
)

result = await Runner.run(calculator_agent, "What is 15% of 2340?")

print("=" * 60)
print("OPENAI AGENTS SDK — WORKER MODEL")
print("=" * 60)
print(result.final_output)
print("=" * 60)

OPENAI AGENTS SDK — WORKER MODEL
15% of 2340 is 351.0.


---

## Framework 2: LangGraph — The Flowchart Model

**Why we chose it:** LangGraph is the production choice. When you need explicit
control over every step — conditional branching, loops, human approval gates,
state persistence — LangGraph gives you that control. It's like having a whiteboard
where you draw the entire workflow before running it.

**Why LangGraph and NOT LangChain?** LangChain is the umbrella library. LangGraph
is the specific framework for building agents as state graphs. LangChain handles
chains, prompts, and retrieval. LangGraph handles agents. We're here to learn agents.

**The mental model:** You're drawing a flowchart:
- **Nodes** = boxes (functions that do work)
- **Edges** = arrows (paths between boxes)
- **State** = data that flows through the boxes
- **Conditional edges** = diamond decision points

**When it wins:** Production workflows, complex routing, loops, stateful agents.
**When it loses:** Quick prototypes where all that graph ceremony is overkill.

In [6]:
# ── LangGraph: The Flowchart Model ──────────────────────
# Two approaches: prebuilt (quick) vs manual graph (full control)

from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

model = get_langgraph_model()

# LangGraph tools use @tool from langchain_core
@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression like '15% of 2340'."""
    import re
    pct_match = re.match(r"(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)", expression)
    if pct_match:
        return str(float(pct_match.group(2)) * float(pct_match.group(1)) / 100)
    try:
        return str(eval(expression))
    except Exception:
        return f"Could not evaluate: {expression}"

# The prebuilt approach — same as Day 1, but WITH tools this time
graph_agent = create_react_agent(model, tools=[calculate])

result = graph_agent.invoke(
    {"messages": [HumanMessage(content="What is 15% of 2340?")]}
)

print("=" * 60)
print("LANGGRAPH — FLOWCHART MODEL (prebuilt)")
print("=" * 60)
print(result["messages"][-1].content)
print("=" * 60)
print(f"Messages in state: {len(result['messages'])}")
print("(Notice: the tool was called. That's an extra message in the state.)")

/var/folders/s0/2pw05fm942n5t1b5j7zzd_5m0000gn/T/ipykernel_18317/3328040703.py:24: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph_agent = create_react_agent(model, tools=[calculate])


LANGGRAPH — FLOWCHART MODEL (prebuilt)
15% of 2340 is 351.0.
Messages in state: 4
(Notice: the tool was called. That's an extra message in the state.)


---

## Framework 3: CrewAI — The Team Model

**Why we chose it:** CrewAI makes multi-agent teams trivially easy. When you think
in terms of "I need a researcher, a writer, and an editor" — CrewAI translates that
directly into code. No graph drawing, no runner setup. Just roles and tasks.

**The mental model:** You're building a team:
- **Agent** = a team member (role, goal, backstory)
- **Task** = work to be done (description, expected output)
- **Crew** = the team (agents, tasks, process)

**When it wins:** Multi-agent teams, content pipelines, role-based collaboration.
**When it loses:** Single-agent tasks (too much ceremony), fine-grained control flow.

In [7]:
# ── CrewAI: The Team Model ──────────────────────────────
# Notice: even a calculator needs a role, goal, and backstory.

from crewai import Agent, Task, Crew, Process

llm = get_crewai_llm()

calc_agent = Agent(
    role="Math Specialist",
    goal="Calculate mathematical expressions accurately",
    backstory="You are a precise mathematician who never makes calculation errors. "
                 "You break down complex expressions step by step.",
    llm=llm,
    verbose=True,
)

calc_task = Task(
    description="Calculate 15% of 2340. Show the result.",
    expected_output="The numerical result of 15% of 2340.",
    agent=calc_agent,
)

crew = Crew(
    agents=[calc_agent],
    tasks=[calc_task],
    process=Process.sequential,
    verbose=True,
)

result = await crew.kickoff_async()

print()
print("=" * 60)
print("CREWAI — TEAM MODEL")
print("=" * 60)
print(result)
print("=" * 60)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d0ecd787-42f1-41bb-ab0b-334690868a27                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Calculate 15% of 2340. Show the result.                                                                  │
│  ID: d3c408a1-9c8b-4127-8762-f19df451a885                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Specialist                                                                                         │
│                                                                                                                 │
│  Task: Calculate 15% of 2340. Show the result.                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Specialist                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  To calculate 15% of 2340, first convert the percentage to a decimal.                                           │
│                                                                                                                 │
│  Step one: Convert percent to decimal.                                                                          │
│  \[ \text{Percent} = 15\% = \frac{15}{100} = 0.15 \]                                                            │
│                                                                                                                 │
│  Now we multiply this decimal by the number 2340.                                                               │
│                                                                                                                 │
│  Step two: Multiply the decimal by the given number.                                                            │
│  \[ 0.15 \times 2340 \]                                                                                         │
│                                                                                                                 │
│  Carry out step two:                                                                                            │
│                                                                                                                 │
│  \[ 0.15 \times 2340 = 0.15 \times (2300 + 40) = (0.15 \times 2300) + (0.15 \times 40) \]                       │
│                                                                                                                 │
│  Calculate each part separately:                                                                                │
│  \[ 0.15 \times 2300 = 345 \]                                                                                   │
│  \[ 0.15 \times 40 = 6 \]                                                                                       │
│                                                                                                                 │
│  Add the two parts together to get the total result.                                                            │
│  \[ 345 + 6 = 351 \]                                                                                            │
│                                                                                                                 │
│  So,                                                                                                            │
│  \[ 15\% \text{ of } 2340 = 351. \]                                                                             │
│                                                                                                                 │
│  The numerical result is \(351\).                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Calculate 15% of 2340. Show the result.                                                                  │
│  Agent: Math Specialist                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d0ecd787-42f1-41bb-ab0b-334690868a27                                                                       │
│  Final Output: To calculate 15% of 2340, first convert the percentage to a decimal.                             │
│                                                                                                                 │
│  Step one: Convert percent to decimal.                                                                          │
│  \[ \text{Percent} = 15\% = \frac{15}{100} = 0.15 \]                                                            │
│                                                                                                                 │
│  Now we multiply this decimal by the number 2340.                                                               │
│                                                                                                                 │
│  Step two: Multiply the decimal by the given number.                                                            │
│  \[ 0.15 \times 2340 \]                                                                                         │
│                                                                                                                 │
│  Carry out step two:                                                                                            │
│                                                                                                                 │
│  \[ 0.15 \times 2340 = 0.15 \times (2300 + 40) = (0.15 \times 2300) + (0.15 \times 40) \]                       │
│                                                                                                                 │
│  Calculate each part separately:                                                                                │
│  \[ 0.15 \times 2300 = 345 \]                                                                                   │
│  \[ 0.15 \times 40 = 6 \]                                                                                       │
│                                                                                                                 │
│  Add the two parts together to get the total result.                                                            │
│  \[ 345 + 6 = 351 \]                                                                                            │
│                                                                                                                 │
│  So,                                                                                                            │
│  \[ 15\% \text{ of } 2340 = 351. \]                                                                             │
│                                                                                                                 │
│  The numerical result is \(351\).                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


CREWAI — TEAM MODEL
To calculate 15% of 2340, first convert the percentage to a decimal. 

Step one: Convert percent to decimal.
\[ \text{Percent} = 15\% = \frac{15}{100} = 0.15 \]

Now we multiply this decimal by the number 2340.

Step two: Multiply the decimal by the given number.
\[ 0.15 \times 2340 \]

Carry out step two:

\[ 0.15 \times 2340 = 0.15 \times (2300 + 40) = (0.15 \times 2300) + (0.15 \times 40) \]

Calculate each part separately:
\[ 0.15 \times 2300 = 345 \]
\[ 0.15 \times 40 = 6 \]

Add the two parts together to get the total result.
\[ 345 + 6 = 351 \]

So,
\[ 15\% \text{ of } 2340 = 351. \]

The numerical result is \(351\).


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Side-by-Side: Why Three, Not Five

You might wonder: why not LangChain? LlamaIndex? Microsoft AutoGen?

| Framework | Why we dropped it | What it does well |
|---|---|---|
| **LangChain** | It's the umbrella, not the agent framework | Chains, retrieval, prompt templates |
| **LlamaIndex** | Specialized for RAG, not general agents | Document indexing, query engines, citations |
| **Microsoft AutoGen** | Enterprise-focused, still evolving, heavy ceremony | Azure integration, enterprise governance |

**Our three cover the full spectrum:**
- **Quick & simple** → OpenAI Agents SDK
- **Full control** → LangGraph
- **Multi-agent teams** → CrewAI

If you master these three, you can pick up any other framework in a day.

---

## Key Takeaway

Frameworks are **mental models**, not just libraries. The code you write reflects how
the framework thinks about agents:

- **Worker** (OpenAI SDK) → name, instructions, tools, go
- **Flowchart** (LangGraph) → nodes, edges, state, compile, run
- **Team** (CrewAI) → roles, goals, backstories, tasks, kickoff

The framework you choose should match how YOU think about the problem.

---
